# 0. ETRI 위키백과 QA API 데이터 수집

본 노트북은 **ETRI 위키백과 QA API**를 사용하여 외부 QA 데이터셋을 수집하고 저장합니다.

**⚠️ 중요:**
- API 일일 호출 제한: 5,000건/일
- 수집된 데이터는 `data/etri_qa_dataset.json`에 저장됩니다
- 이 노트북은 **한 번만 실행**하면 됩니다
- 02, 03 노트북에서는 저장된 데이터를 불러와서 사용합니다


## 0.1. 환경 설정


In [1]:
import os
import sys
import json
import urllib3
import time
from pathlib import Path
from typing import Dict, List, Optional
from tqdm.auto import tqdm
from datasets import Dataset, DatasetDict, load_from_disk

# 프로젝트 루트 경로 설정
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))

# 데이터 저장 경로
data_dir = Path().resolve() / "data"
data_dir.mkdir(parents=True, exist_ok=True)

print(f"프로젝트 루트: {project_root}")
print(f"데이터 저장 경로: {data_dir}")


d:\Repos\pro-nlp-mrc-nlp-01\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


프로젝트 루트: D:\Repos\pro-nlp-mrc-nlp-01
데이터 저장 경로: D:\Repos\pro-nlp-mrc-nlp-01\notebooks\external_data_set\data


## 0.2. ETRI API 설정

⚠️ **API 키를 아래에 입력하세요!**


In [2]:
# ETRI API 설정
ETRI_API_URL = "http://epretx.etri.re.kr:8000/api/WikiQA/"
ETRI_ACCESS_KEY = "d4a79a0b-64a4-432a-8a9a-24fd5bf433e4"  # 여기에 API 키 입력

# 수집 설정
NUM_QUESTIONS = 1000  # 수집할 질문 수 (최대 5000/일)
MIN_CONFIDENCE = 0.5  # 최소 신뢰도 임계값
API_DELAY = 0.2  # API 호출 간 딜레이 (초)

print(f"API URL: {ETRI_API_URL}")
print(f"수집 예정 질문 수: {NUM_QUESTIONS}")
print(f"최소 신뢰도: {MIN_CONFIDENCE}")


API URL: http://epretx.etri.re.kr:8000/api/WikiQA/
수집 예정 질문 수: 1000
최소 신뢰도: 0.5


## 0.3. ETRI API 클라이언트


In [3]:
class ETRIWikiQA:
    """ETRI 위키백과 QA API 클라이언트"""
    
    def __init__(self, access_key: str):
        self.api_url = ETRI_API_URL
        self.access_key = access_key
        self.http = urllib3.PoolManager(cert_reqs='CERT_NONE')
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    def query(self, question: str, engine_type: str = "hybridqa") -> Dict:
        """위키백과 QA API 호출"""
        request_json = {
            "argument": {
                "question": question,
                "type": engine_type
            }
        }
        
        try:
            response = self.http.request(
                "POST",
                self.api_url,
                headers={
                    "Content-Type": "application/json; charset=UTF-8",
                    "Authorization": self.access_key
                },
                body=json.dumps(request_json)
            )
            
            if response.status == 200:
                return json.loads(response.data.decode('utf-8'))
            else:
                return None
                
        except Exception as e:
            return None
    
    def extract_qa_data(self, response: Dict) -> Optional[Dict]:
        """API 응답에서 QA 데이터 추출"""
        if not response or response.get('result') != 0:
            return None
        
        try:
            return_object = response.get('return_object', {})
            wiki_info = return_object.get('WiKiInfo', {})
            
            answer_info = wiki_info.get('AnswerInfo', [])
            if not answer_info:
                return None
            
            best_answer = answer_info[0]
            answer = best_answer.get('answer', '')
            confidence = best_answer.get('confidence', 0)
            
            ir_info = wiki_info.get('IRInfo', [])
            context = ""
            wiki_title = ""
            if ir_info:
                context = ir_info[0].get('sent', '')
                wiki_title = ir_info[0].get('wiki_title', '')
            
            return {
                'answer': answer,
                'confidence': confidence,
                'context': context,
                'wiki_title': wiki_title
            }
            
        except Exception as e:
            return None

# API 클라이언트 초기화
etri_qa = ETRIWikiQA(ETRI_ACCESS_KEY)
print("✅ ETRI API 클라이언트 초기화 완료")


✅ ETRI API 클라이언트 초기화 완료


## 0.4. API 연결 테스트


In [4]:
# API 테스트
test_question = "대한민국의 수도는 어디인가요?"
print(f"테스트 질문: {test_question}")
print("-" * 50)

response = etri_qa.query(test_question)
if response:
    qa_data = etri_qa.extract_qa_data(response)
    if qa_data:
        print("✅ API 연결 성공!")
        print(f"정답: {qa_data['answer']}")
        print(f"신뢰도: {qa_data['confidence']:.4f}")
    else:
        print("❌ 데이터 추출 실패")
else:
    print("❌ API 연결 실패")


테스트 질문: 대한민국의 수도는 어디인가요?
--------------------------------------------------
✅ API 연결 성공!
정답: 서울특별시
신뢰도: 2.0427


## 0.5. 기존 데이터셋에서 질문 로드


In [5]:
# 기존 데이터셋에서 질문 로드
original_data_path = project_root / "data" / "train_dataset"

if original_data_path.exists():
    original_datasets = load_from_disk(str(original_data_path))
    original_questions = original_datasets['train']['question']
    print(f"✅ 원본 데이터셋에서 {len(original_questions)}개의 질문 로드")
else:
    print("❌ 원본 데이터셋을 찾을 수 없습니다.")
    original_questions = []


✅ 원본 데이터셋에서 3952개의 질문 로드


## 0.6. ETRI API로 데이터 수집

⚠️ **이 셀은 시간이 오래 걸릴 수 있습니다 (약 3-5분)**


In [6]:
def collect_etri_qa_data(
    questions: List[str],
    etri_client: ETRIWikiQA,
    num_questions: int = 1000,
    min_confidence: float = 0.5,
    delay: float = 0.2
) -> List[Dict]:
    """ETRI API를 사용하여 QA 데이터 수집"""
    collected_data = []
    failed_count = 0
    
    sample_questions = questions[:num_questions]
    
    for idx, question in enumerate(tqdm(sample_questions, desc="데이터 수집")):
        try:
            response = etri_client.query(question, "hybridqa")
            
            if response:
                qa_data = etri_client.extract_qa_data(response)
                
                if qa_data and qa_data['confidence'] >= min_confidence:
                    answer = qa_data['answer']
                    context = qa_data['context']
                    answer_start = context.find(answer)
                    
                    if answer_start != -1:
                        collected_data.append({
                            'id': f"etri-{idx:05d}",
                            'question': question,
                            'context': context,
                            'answers': {
                                'text': [answer],
                                'answer_start': [answer_start]
                            },
                            'title': qa_data['wiki_title'],
                            'confidence': qa_data['confidence']
                        })
                else:
                    failed_count += 1
            else:
                failed_count += 1
            
            time.sleep(delay)
            
        except Exception as e:
            failed_count += 1
            continue
    
    print(f"\n✅ 수집 완료!")
    print(f"   - 성공: {len(collected_data)}개")
    print(f"   - 실패/스킵: {failed_count}개")
    
    return collected_data

# 데이터 수집 실행
if original_questions:
    print(f"📥 ETRI API로 {NUM_QUESTIONS}개의 질문에 대한 데이터 수집 시작...")
    print(f"   예상 소요 시간: 약 {NUM_QUESTIONS * API_DELAY / 60:.1f}분\n")
    
    etri_qa_data = collect_etri_qa_data(
        questions=original_questions,
        etri_client=etri_qa,
        num_questions=NUM_QUESTIONS,
        min_confidence=MIN_CONFIDENCE,
        delay=API_DELAY
    )
else:
    print("❌ 질문 데이터가 없어 수집을 건너뜁니다.")
    etri_qa_data = []


📥 ETRI API로 1000개의 질문에 대한 데이터 수집 시작...
   예상 소요 시간: 약 3.3분



데이터 수집: 100%|██████████| 1000/1000 [30:02<00:00,  1.80s/it]


✅ 수집 완료!
   - 성공: 195개
   - 실패/스킵: 652개


## 0.7. 수집된 데이터 저장


In [7]:
# 수집된 데이터 저장
etri_data_path = data_dir / "etri_qa_dataset.json"

if etri_qa_data:
    with open(etri_data_path, 'w', encoding='utf-8') as f:
        json.dump(etri_qa_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 데이터 저장 완료!")
    print(f"   - 저장 경로: {etri_data_path}")
    print(f"   - 총 샘플 수: {len(etri_qa_data)}개")
else:
    print("❌ 저장할 데이터가 없습니다.")


✅ 데이터 저장 완료!
   - 저장 경로: D:\Repos\pro-nlp-mrc-nlp-01\notebooks\external_data_set\data\etri_qa_dataset.json
   - 총 샘플 수: 195개


## 0.8. 수집된 데이터 확인


In [8]:
# 수집된 데이터 샘플 확인
if etri_qa_data:
    print("=== 수집된 데이터 샘플 ===\n")
    for i, sample in enumerate(etri_qa_data[:3]):
        print(f"[샘플 {i+1}]")
        print(f"  ID: {sample['id']}")
        print(f"  Question: {sample['question']}")
        print(f"  Answer: {sample['answers']['text'][0]}")
        print(f"  Context: {sample['context'][:100]}...")
        print(f"  Confidence: {sample['confidence']:.4f}")
        print()
    
    # 통계
    import numpy as np
    confidences = [d['confidence'] for d in etri_qa_data]
    print("=== 통계 ===")
    print(f"총 샘플 수: {len(etri_qa_data)}")
    print(f"평균 신뢰도: {np.mean(confidences):.4f}")
    print(f"최소 신뢰도: {np.min(confidences):.4f}")
    print(f"최대 신뢰도: {np.max(confidences):.4f}")


=== 수집된 데이터 샘플 ===

[샘플 1]
  ID: etri-00001
  Question: 현대적 인사조직관리의 시발점이 된 책은?
  Answer: 경영의 실제
  Context: 인사조직관리의 역사 현대적 경영 이론 시대의 인사조직관리 '근대적 경영학' 또는 '고전적 경영학'에서 현대적 경영학으로 전환되는 시기는 1950년대이다. 2차 세계대전을 마치고, ...
  Confidence: 1.2558

[샘플 2]
  ID: etri-00020
  Question: 항우가 진나라를 멸하면서 조왕 헐은 어디로 이동했는가?
  Answer: 대나라
  Context: 생애 대나라 왕 기원전 206년에 항우가 진나라를 멸하고 각지에 제후왕들을 봉하면서 원래의 육국의 세력을 약화시키고자 각 나라를 쪼갰고, 자신을 따라온 장수들을 중용해 각각의 본국...
  Confidence: 0.7083

[샘플 3]
  ID: etri-00028
  Question: 카르텔 내부의 세력 다툼이 일어나기 전 카르텔의 리더는 누구였나?
  Answer: 아르투로
  Context: 주요 카르텔 벨뜨란-레이바 카르텔 마르코스 아르투로, 카를로스, 알프레도, 그리고 엑토르 벨트란 레이바의 4형제가 조직한 마약 범죄 카르텔이다. 2004 ~ 2005년, 아르투로 ...
  Confidence: 1.6234

=== 통계 ===
총 샘플 수: 195
평균 신뢰도: 1.4612
최소 신뢰도: 0.5200
최대 신뢰도: 3.8226


## 완료!

이제 `02_etri_qa_experiment.ipynb`와 `03_combined_experiment.ipynb`에서 저장된 데이터를 불러와서 사용할 수 있습니다.
